# 🎵 Average Duration of WebM Audio Files
Works with **Chrome-recorded WebM files** that have `Duration: N/A` in their headers.
Since Chrome's `MediaRecorder` does not write duration metadata, the file is fully decoded
through `ffmpeg`'s null muxer to measure the actual playback length.

In [ ]:
!ffprobe -version 2>&1 | head -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✏️  Change this to your folder path
WEBM_FOLDER = '/content/drive/MyDrive/HMS/Speach_to_JSON/Models benchmarking/Data'

# ✏️  Add filenames to skip (just the filename, not the full path)
EXCEPTIONS = [
    # 'session_abc123_example_000.webm',
    # 'session_abc123_example_001.webm',
]

In [ ]:
import subprocess, re
from pathlib import Path


def get_duration(filepath: str) -> float | None:
    """
    Decode the entire file through ffmpeg's null muxer and read the final
    timestamp. This is the only reliable method for Chrome MediaRecorder
    WebM files, which store Duration: N/A in their headers.
    """
    cmd = ['ffmpeg', '-y', '-i', str(filepath), '-f', 'null', '-']
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        matches = re.findall(r'time=(\d+):(\d+):(\d+\.\d+)', r.stderr)
        if matches:
            h, m, s = matches[-1]
            return int(h) * 3600 + int(m) * 60 + float(s)
    except Exception as e:
        print(f'    ERROR: {e}')
    return None


def fmt(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    return f'{m}m {s:02d}s  ({seconds:.2f}s)'


# ---------- Scan ----------
folder = Path(WEBM_FOLDER)
if not folder.exists():
    raise FileNotFoundError(f'Folder not found: {folder}')

exceptions_set = set(EXCEPTIONS)
webm_files = sorted(folder.rglob('*.webm'))

print(f'📂 {folder}')
print(f'🔍 Found {len(webm_files)} .webm file(s)  |  Exceptions: {len(exceptions_set)}')
print('   (Decoding each file to measure duration — this may take a minute...)\n')

durations, names, failed, skipped = [], [], [], []

for i, f in enumerate(webm_files, 1):
    if f.name in exceptions_set:
        skipped.append(f.name)
        print(f'  [{i:>3}/{len(webm_files)}] ⏭️  {f.name:<52} skipped')
        continue

    d = get_duration(f)
    if d is not None:
        durations.append(d)
        names.append(f.name)
        print(f'  [{i:>3}/{len(webm_files)}] ✅  {f.name:<52} {fmt(d)}')
    else:
        failed.append(f.name)
        print(f'  [{i:>3}/{len(webm_files)}] ❌  {f.name}  — failed')

# ---------- Summary ----------
print()
if durations:
    avg     = sum(durations) / len(durations)
    total   = sum(durations)
    min_idx = durations.index(min(durations))
    max_idx = durations.index(max(durations))
    print('━' * 65)
    print(f'  Files processed : {len(durations)}  |  Skipped: {len(skipped)}  |  Failed: {len(failed)}')
    print(f'  Shortest        : {fmt(min(durations))}')
    print(f'                    👉  {names[min_idx]}')
    print(f'  Longest         : {fmt(max(durations))}')
    print(f'                    👉  {names[max_idx]}')
    print(f'  Total           : {fmt(total)}')
    print(f'  ⭐ Average       : {fmt(avg)}')
    print('━' * 65)
else:
    print('⚠️  No durations could be read from any file.')